# Phase 16 — Component ablation and multi-seed check

Companion to `15_ed_es_final_model.ipynb`. Two analyses:

1. **Part A -- component-wise ablation** (frame selection vs. +contraction-delta vs. +Quality-FiLM), disentangling the bundled Stage 4 to Stage 5 comparison.
2. **Part B -- multi-seed variance** of the final (SimCLR-init, delta+FiLM) configuration.

The selective-prediction table for the final model is produced separately by `16c_selective_prediction_final_model.ipynb`.

**Test-set discipline.** Part A's two new architectures (ED/ES-alone, +delta only) are evaluated on **validation only**. The full model (+delta+FiLM) reuses the test numbers already obtained in notebook 15 (MAE 6.052, AUC 0.921) rather than re-touching the test set. Part B re-trains the already-frozen final architecture with different seeds, so evaluating those runs on test is variance checking, not architecture selection.

The notebook is self-contained: it re-mounts Drive and reloads the data splits. Run cells top to bottom with a GPU runtime.

## Setup — imports, Drive mount, data splits (same as notebook 15)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
EDES_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/edes', 'frames')
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')
PRETRAINED_BACKBONE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'simclr_pretrain_v1', 'backbone_only.pt')

assert os.path.isdir(EDES_CACHE_DIR) and len(os.listdir(EDES_CACHE_DIR)) > 9000, \
    'Run 14_ed_es_extraction.ipynb to completion first.'

df = pd.read_csv(FULL_CSV)
quality_df = pd.read_csv(quality_csv_path)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
HFrEF_THRESHOLD = 40

for col in ['sharpness', 'contrast']:
    lo, hi = quality_df[col].quantile(0.02), quality_df[col].quantile(0.98)
    quality_df[col + '_norm'] = ((quality_df[col] - lo) / max(hi - lo, 1e-6)).clip(0, 1)
quality_df['quality_score'] = quality_df[['sharpness_norm', 'contrast_norm']].mean(axis=1)
df = df.merge(quality_df[[filename_col, 'quality_score']], on=filename_col, how='left')

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(EDES_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask & df['quality_score'].notna()].reset_index(drop=True)
train_df = clean_df[clean_df['Split'].str.upper() == 'TRAIN'].reset_index(drop=True)
val_df = clean_df[clean_df['Split'].str.upper() == 'VAL'].reset_index(drop=True)
test_df = clean_df[clean_df['Split'].str.upper() == 'TEST'].reset_index(drop=True)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU before running this notebook (Runtime > Change runtime type).')

## Dataset class and loaders (same as notebook 15)

In [ ]:
class EchoEDESDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        self.train = train
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        pair = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        pair = torch.from_numpy(pair).permute(0, 3, 1, 2).float() / 255.0
        pair = F.interpolate(pair, size=(112, 112), mode='bilinear', align_corners=False)
        if self.train and torch.rand(1).item() < 0.5:
            pair = torch.flip(pair, dims=[3])
        pair = (pair - self.mean) / self.std
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        quality = float(row['quality_score']) if not pd.isna(row['quality_score']) else 0.5
        return pair, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

BATCH_SIZE = 24
train_ds = EchoEDESDataset(train_df, EDES_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoEDESDataset(val_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
test_ds = EchoEDESDataset(test_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Shared: flexible model (toggle delta / FiLM) and a reusable train+eval loop

In [ ]:
class QualityFiLM(nn.Module):
    def __init__(self, feat_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 2 * feat_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.feat_dim = feat_dim

    def forward(self, quality_score):
        params = self.net(quality_score.unsqueeze(-1))
        gamma, beta = params[:, :self.feat_dim], params[:, self.feat_dim:]
        return 1.0 + gamma, beta

class EchoEDESCQRAblatable(nn.Module):
    """Same architecture as notebook 15's EchoEDESCQR, with use_delta / use_film
    toggles for the component-wise ablation."""
    def __init__(self, dropout_p=0.2, pretrained_backbone_path=None,
                 use_delta=True, use_film=True):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()
        if pretrained_backbone_path:
            backbone.load_state_dict(torch.load(pretrained_backbone_path, map_location='cpu'))
        self.backbone = backbone
        self.use_delta = use_delta
        self.use_film = use_film
        combined_dim = 512 * 3 if use_delta else 512 * 2
        self.film = QualityFiLM(feat_dim=combined_dim) if use_film else None
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_mean_head = nn.Linear(combined_dim, 1)
        self.ef_q_lo_head = nn.Linear(combined_dim, 1)
        self.ef_q_hi_head = nn.Linear(combined_dim, 1)
        self.cls_head = nn.Linear(combined_dim, 1)

    def forward(self, pair, quality_score):
        ed = pair[:, 0]
        es = pair[:, 1]
        f_ed = self.backbone(ed)
        f_es = self.backbone(es)
        feats = torch.cat([f_ed, f_es, f_ed - f_es], dim=1) if self.use_delta else torch.cat([f_ed, f_es], dim=1)
        if self.use_film:
            gamma, beta = self.film(quality_score)
            feats = gamma * feats + beta
        feats = self.dropout(feats)
        ef_mean = self.ef_mean_head(feats).squeeze(-1)
        q_lo = self.ef_q_lo_head(feats).squeeze(-1)
        q_hi = self.ef_q_hi_head(feats).squeeze(-1)
        cls_logit = self.cls_head(feats).squeeze(-1)
        return ef_mean, q_lo, q_hi, cls_logit

def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return torch.mean(torch.max(tau * diff, (tau - 1) * diff))

TAU_LO, TAU_HI = 0.05, 0.95

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def evaluate_simple(model, loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_mean.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    return {'mae': mae, 'auc': auc}

def predict_all(model, loader):
    model.eval()
    ef_true_all, ef_mean_all, q_lo_all, q_hi_all, cls_true_all, cls_prob_all = [], [], [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_mean_all.extend(ef_mean.cpu().numpy())
            q_lo_all.extend(q_lo.cpu().numpy()); q_hi_all.extend(q_hi.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    return {
        'ef_true': np.array(ef_true_all), 'ef_mean': np.array(ef_mean_all),
        'q_lo': np.array(q_lo_all), 'q_hi': np.array(q_hi_all),
        'cls_true': np.array(cls_true_all), 'cls_prob': np.array(cls_prob_all),
    }

def train_one_config(use_delta, use_film, seed, n_epochs=10, backbone_path=None):
    """Trains one configuration from scratch; returns (model, val_metrics)."""
    set_seed(seed)
    model = EchoEDESCQRAblatable(pretrained_backbone_path=backbone_path,
                                  use_delta=use_delta, use_film=use_film).to(device)
    with torch.no_grad():
        mean_ef = float(train_df[ef_col].mean()); std_ef = float(train_df[ef_col].std())
        model.ef_mean_head.bias.fill_(mean_ef); model.ef_mean_head.weight.zero_()
        model.ef_q_lo_head.bias.fill_(mean_ef - std_ef); model.ef_q_lo_head.weight.zero_()
        model.ef_q_hi_head.bias.fill_(mean_ef + std_ef); model.ef_q_hi_head.weight.zero_()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    pos_rate = (train_df[ef_col] < HFrEF_THRESHOLD).mean()
    pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
    cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(n_epochs):
        model.train()
        for pair, ef_true, cls_true, quality in train_loader:
            pair, ef_true, cls_true, quality = pair.to(device), ef_true.to(device), cls_true.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            loss = (F.l1_loss(ef_mean, ef_true)
                    + pinball_loss(ef_true, q_lo, TAU_LO)
                    + pinball_loss(ef_true, q_hi, TAU_HI)
                    + cls_criterion(cls_logit, cls_true))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        val_metrics = evaluate_simple(model, val_loader)
        print(f'  [delta={use_delta} film={use_film} seed={seed}] epoch {epoch}: '
              f'val_mae={val_metrics["mae"]:.3f} val_auc={val_metrics["auc"]:.3f}')
    return model, val_metrics

# NOTE ON RUNTIME: each full config trains for N_EPOCHS epochs, same cost as notebook 15's
# main training cell. Part A runs 2 new configs, Part B runs 2 additional seeds of the full
# config -- budget accordingly (this is the 'few GPU-hours' scale claimed in Section 3.7,
# times ~4 for these extra runs). Consider lowering n_epochs for a quick pilot first.

## Part A — Component-wise ablation

Three configurations, all SimCLR-initialized, evaluated on **validation** (test reserved):
1. `ED/ES-alone`: concatenate `[f_ED, f_ES]` only -- no delta, no FiLM.
2. `+delta`: add the contraction-delta feature -- no FiLM.
3. `+delta+FiLM` (= the paper's final model): reuse the existing **test** numbers from notebook 15 (6.052 / 0.921) rather than re-evaluating on test.

In [ ]:
ablation_results = {}

print('Config 1/2: ED/ES-alone (no delta, no FiLM)')
_, val_metrics_edes_alone = train_one_config(
    use_delta=False, use_film=False, seed=42, n_epochs=10,
    backbone_path=PRETRAINED_BACKBONE_PATH)
ablation_results['ED/ES-alone'] = val_metrics_edes_alone

print('Config 2/2: +delta (no FiLM)')
_, val_metrics_plus_delta = train_one_config(
    use_delta=True, use_film=False, seed=42, n_epochs=10,
    backbone_path=PRETRAINED_BACKBONE_PATH)
ablation_results['+delta'] = val_metrics_plus_delta

# Reuse the already-reported TEST numbers for the full final model instead of re-touching test.
ablation_results['+delta+FiLM (final, TEST not VAL)'] = {'mae': 6.052, 'auc': 0.921}

print()
print('Component-wise ablation summary (first two rows = VAL, last row = TEST for reference):')
for k, v in ablation_results.items():
    print(f'  {k:35s}  MAE={v["mae"]:.3f}  AUC={v["auc"]:.3f}')

with open(os.path.join(PROJECT_ROOT, 'logs', 'component_ablation_results.json'), 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('Saved to logs/component_ablation_results.json -- component-wise ablation results.')

## Part B — Multi-seed variance on the final configuration

Two **additional** seeds (the notebook-15 run is implicitly seed "as-run"/unrecorded -- treat it as run 1). Evaluated on **test**: this is standard variance-checking on an already-frozen architecture, not architecture search, so it doesn't carry the same test-discipline concern as Part A.

In [ ]:
seed_results = []
for seed in [123, 2024]:
    model_s, _ = train_one_config(use_delta=True, use_film=True, seed=seed, n_epochs=10,
                                   backbone_path=PRETRAINED_BACKBONE_PATH)
    # CQR calibration on a fresh val split, same procedure as notebook 15 cell 9
    val_results_s = predict_all(model_s, val_loader)
    rng = np.random.default_rng(seed)
    n_val = len(val_results_s['ef_true'])
    perm = rng.permutation(n_val)
    calib_idx = perm[:n_val // 2]
    ALPHA = 0.1
    E_calib = np.maximum(val_results_s['q_lo'][calib_idx] - val_results_s['ef_true'][calib_idx],
                          val_results_s['ef_true'][calib_idx] - val_results_s['q_hi'][calib_idx])
    Q_hat_s = np.quantile(E_calib, min(1.0, (1 - ALPHA) * (1 + 1 / len(calib_idx))))

    test_results_s = predict_all(model_s, test_loader)
    mae_s = mean_absolute_error(test_results_s['ef_true'], test_results_s['ef_mean'])
    auc_s = roc_auc_score(test_results_s['cls_true'], test_results_s['cls_prob'])
    lo_s = test_results_s['q_lo'] - Q_hat_s; hi_s = test_results_s['q_hi'] + Q_hat_s
    cov_s = ((test_results_s['ef_true'] >= lo_s) & (test_results_s['ef_true'] <= hi_s)).mean()
    seed_results.append({'seed': seed, 'test_mae': mae_s, 'test_auc': auc_s, 'test_coverage': cov_s})
    print(f'seed={seed}: TEST MAE={mae_s:.3f} AUC={auc_s:.3f} coverage={cov_s*100:.1f}%')

# Combine with the notebook-15 run (seed unrecorded -> call it 'run 1')
all_runs = [{'seed': 'run1 (notebook 15)', 'test_mae': 6.052, 'test_auc': 0.921, 'test_coverage': 0.893}] + seed_results
maes = [r['test_mae'] for r in all_runs]; aucs = [r['test_auc'] for r in all_runs]
print(f'\nAcross {len(all_runs)} seeds: MAE = {np.mean(maes):.3f} +/- {np.std(maes):.3f}, '
      f'AUC = {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}')

with open(os.path.join(PROJECT_ROOT, 'logs', 'multiseed_results.json'), 'w') as f:
    json.dump(all_runs, f, indent=2)
print('Saved to logs/multiseed_results.json -- mean +/- std across runs.')